# Local Output Modifiers by Location

Computes per-location output modifiers for each good using the goods Ã— attributes matrix (goods_output_modifiers.xlsx). Not parsed from game filesâ€”derived from the Excel matrix and location attributes.

For each location: `local_{good}_output_modifier` = sum of matrix cells (topography + vegetation + climate + water). **Base output is 1.0.** Final output multiplier = `1.0 + local_{good}_output_modifier`. We also store `local_{good}_output_multiplier` = 1.0 + modifier for direct use.

In [1]:
import pandas as pd

from analysis.building_levels.building_analysis import CapacityAnalyzer, load_goods_output_modifiers

In [2]:
analyzer = CapacityAnalyzer()
df_locations = analyzer.location_data.get_merged_df()
matrix = load_goods_output_modifiers()

goods = list(matrix.index)

Loading live game data...


In [3]:
def compute_output_modifier(row, good, matrix_df):
    """Sum applicable matrix cells for this location and good."""
    topo = row.get("topography") or ""
    veg = row.get("vegetation") or ""
    climate = row.get("climate") or ""
    
    total = 0.0
    total += matrix_df.loc[good].get(topo, 0.0)
    total += matrix_df.loc[good].get(veg, 0.0)
    total += matrix_df.loc[good].get(climate, 0.0)
    
    if str(row.get("has_river", "no")).lower() == "yes":
        total += matrix_df.loc[good].get("has_river", 0.0)
    if str(row.get("is_adjacent_to_lake", "no")).lower() == "yes":
        total += matrix_df.loc[good].get("is_adjacent_to_lake", 0.0)
    if str(row.get("is_coastal", "no")).lower() == "yes":
        total += matrix_df.loc[good].get("is_coastal", 0.0)
    
    return total

for good in goods:
    col = f"local_{good}_output_modifier"
    df_locations[col] = df_locations.apply(
        lambda row, g=good: compute_output_modifier(row, g, matrix),
        axis=1,
    )
    # Base output = 1.0; add our sum to get the actual output multiplier
    df_locations[f"local_{good}_output_multiplier"] = 1.0 + df_locations[col]

## Output modifier stats (min, max, median, etc.)

In [4]:
mod_cols = [f"local_{g}_output_modifier" for g in goods if f"local_{g}_output_modifier" in df_locations.columns]
df_modifier_stats = df_locations[mod_cols].describe().T.drop(columns=["count"], errors="ignore")

# Color-coded for readability: gradient per column, high-contrast, larger font
df_modifier_stats.style.background_gradient(axis=0, cmap="RdYlGn").set_properties(
    **{"font-size": "13px", "font-weight": "500"}
).set_table_styles(
    [
        {"selector": "th", "props": [("font-size", "14px"), ("font-weight", "bold"), ("background", "#333"), ("color", "white")]},
        {"selector": "td", "props": [("padding", "8px")]},
    ]
)

,mean,std,min,25%,50%,75%,max
local_fruit_output_modifier,-0.049316,0.448339,-1.500000,-0.350000,0.150000,0.300000,0.700000
local_fish_output_modifier,-0.122638,0.371444,-1.600000,-0.300000,0.050000,0.070000,0.500000
local_wool_output_modifier,-0.071198,0.496728,-1.800000,-0.400000,0.000000,0.350000,0.700000
local_livestock_output_modifier,-0.189507,0.548911,-2.000000,-0.550000,-0.100000,0.250000,0.700000
local_millet_output_modifier,0.069683,0.393023,-1.300000,-0.200000,0.200000,0.350000,0.550000
local_wheat_output_modifier,-0.170525,0.563476,-1.900000,-0.700000,-0.150000,0.300000,0.750000
local_maize_output_modifier,-0.066337,0.441939,-1.700000,-0.300000,0.050000,0.200000,0.650000
local_rice_output_modifier,-0.124803,0.623904,-2.300000,-0.650000,0.150000,0.350000,0.750000
local_legumes_output_modifier,0.082888,0.352977,-1.400000,-0.050000,0.150000,0.350000,0.700000
local_potato_output_modifier,-0.065700,0.455395,-1.600000,-0.400000,0.050000,0.350000,0.600000


## Sample locations with output multipliers (1.0 + modifier)

## Location Power Potential (Capacity × Best Good Multiplier)

For each building: capacity × best output multiplier among its goods (cap at 0 if best < 0). Two variants: **with pop/dev** (full capacity) and **without pop/dev** (base + env + rank only). Sum across buildings = total_rural_potential and total_rural_potential_fixed.

In [5]:
# Building → goods mapping (from pp_building_capacity_values)
BUILDING_GOODS = {
    "fruit_orchard": ["fruit"],
    "fishing_village": ["fish"],
    "sheep_farms": ["wool"],
    "farming_village": ["livestock", "millet", "wheat", "maize", "rice", "legumes", "potato", "olives"],
    "forest_village": ["leather", "wild_game", "fur"],
}
# Display name → building_id for analyzer output
BUILDING_NAME_TO_ID = {
    "Fruit Orchard": "fruit_orchard",
    "Fishing Village": "fishing_village",
    "Sheep Farm": "sheep_farms",
    "Farming Village": "farming_village",
    "Forest Village": "forest_village",
}

# Get capacity per location per building
df_cap = analyzer.calculate_capacities_for_locations(df_locations)
# Full capacity (includes pop + dev bonus)
df_cap_pivot = df_cap.pivot_table(index="Location", columns="Building", values="Total Bonus", aggfunc="first")
# Fixed capacity (base + env + rank only, no pop/dev). Zero when env < 1 (not buildable).
df_cap["Fixed Bonus"] = df_cap.apply(lambda r: r["Base"] + r["Environmental Bonus"] + r["Rank Bonus"] if r["Environmental Bonus"] >= 1 else 0.0, axis=1)
df_cap_fixed_pivot = df_cap.pivot_table(index="Location", columns="Building", values="Fixed Bonus", aggfunc="first")

# For each building: best_mult = max(0, max(local_{good}_output_multiplier))
# Then potential = capacity × best_mult (both full and fixed)
for b_name, building_id in BUILDING_NAME_TO_ID.items():
    goods_list = BUILDING_GOODS[building_id]
    mult_cols = [f"local_{g}_output_multiplier" for g in goods_list if f"local_{g}_output_multiplier" in df_locations.columns]
    if not mult_cols:
        continue
    best_mult = df_locations[mult_cols].max(axis=1).clip(lower=0.0)
    # Full potential (with pop + dev)
    cap_series = (
        df_locations["location"].map(df_cap_pivot[b_name].to_dict()).fillna(0.0)
        if b_name in df_cap_pivot.columns
        else pd.Series(0.0, index=df_locations.index)
    )
    df_locations[f"{building_id}_potential"] = cap_series * best_mult
    # Fixed potential (without pop + dev)
    cap_fixed_series = (
        df_locations["location"].map(df_cap_fixed_pivot[b_name].to_dict()).fillna(0.0)
        if b_name in df_cap_fixed_pivot.columns
        else pd.Series(0.0, index=df_locations.index)
    )
    df_locations[f"{building_id}_potential_fixed"] = cap_fixed_series * best_mult

# Total potential: both full (with pop/dev) and fixed (without)
potential_cols = [f"{bid}_potential" for bid in BUILDING_GOODS if f"{bid}_potential" in df_locations.columns]
potential_fixed_cols = [f"{bid}_potential_fixed" for bid in BUILDING_GOODS if f"{bid}_potential_fixed" in df_locations.columns]
df_locations["total_rural_potential"] = df_locations[potential_cols].sum(axis=1)
df_locations["total_rural_potential_fixed"] = df_locations[potential_fixed_cols].sum(axis=1)

### Sample and top locations by total_rural_potential

In [6]:
potential_cols = [f"{bid}_potential" for bid in BUILDING_GOODS if f"{bid}_potential" in df_locations.columns]
potential_fixed_cols = [f"{bid}_potential_fixed" for bid in BUILDING_GOODS if f"{bid}_potential_fixed" in df_locations.columns]
base_cols = ["location", "region", "topography", "vegetation", "climate"]
available = [c for c in base_cols if c in df_locations.columns]
# With pop + dev (full capacity)
cols_full = available + potential_cols + ["total_rural_potential"]
cols_full = [c for c in cols_full if c in df_locations.columns]
print("With pop + dev:")
display(df_locations[cols_full].head(20))
# Without pop + dev (fixed capacity)
cols_fixed = available + potential_fixed_cols + ["total_rural_potential_fixed"]
cols_fixed = [c for c in cols_fixed if c in df_locations.columns]
print("Without pop + dev (fixed):")
display(df_locations[cols_fixed].head(20))

With pop + dev:


,location,region,topography,vegetation,climate,fruit_orchard_potential,fishing_village_potential,sheep_farms_potential,farming_village_potential,forest_village_potential,total_rural_potential
0,stockholm,scandinavian_region,flatland,grasslands,continental,3.0680,10.8680,10.4940,12.5120,3.6580,40.6000
1,norrtalje,scandinavian_region,flatland,forest,continental,0.0000,10.8284,5.9740,8.5680,8.5680,33.9384
2,enkoping,scandinavian_region,flatland,grasslands,continental,4.4875,0.0000,12.5235,19.7030,5.5645,42.2785
3,uppsala,scandinavian_region,flatland,farmland,continental,5.3595,0.0000,9.2535,22.6975,5.9550,43.2655
4,kastelholm,scandinavian_region,flatland,grasslands,continental,4.6280,20.2280,12.4740,14.5520,5.5180,57.4000
5,tierp,scandinavian_region,flatland,woods,continental,0.0000,10.6785,6.2550,9.5635,7.2380,33.7350
6,heby,scandinavian_region,flatland,grasslands,continental,4.3125,0.0000,12.2925,14.3650,11.5475,42.5175
7,nykoping,scandinavian_region,flatland,grasslands,continental,4.8490,20.4490,12.7545,14.8410,5.7815,58.6750
8,kolmarden,scandinavian_region,flatland,forest,continental,0.0000,0.0000,5.8435,8.4420,8.4420,22.7275
9,strangnas,scandinavian_region,flatland,woods,continental,0.0000,0.0000,6.2700,9.5790,7.2520,23.1010


Without pop + dev (fixed):


,location,region,topography,vegetation,climate,fruit_orchard_potential_fixed,fishing_village_potential_fixed,sheep_farms_potential_fixed,farming_village_potential_fixed,forest_village_potential_fixed,total_rural_potential_fixed
0,stockholm,scandinavian_region,flatland,grasslands,continental,-1.30,6.50,4.95,6.8,-1.55,15.40
1,norrtalje,scandinavian_region,flatland,forest,continental,0.00,8.56,2.90,5.6,5.60,22.66
2,enkoping,scandinavian_region,flatland,grasslands,continental,1.25,0.00,8.25,15.3,1.55,26.35
3,uppsala,scandinavian_region,flatland,farmland,continental,1.35,0.00,4.65,17.5,1.50,25.00
4,kastelholm,scandinavian_region,flatland,grasslands,continental,1.30,16.90,8.25,10.2,1.55,38.20
5,tierp,scandinavian_region,flatland,woods,continental,0.00,8.40,3.00,6.2,4.20,21.80
6,heby,scandinavian_region,flatland,grasslands,continental,1.25,0.00,8.25,10.2,7.75,27.45
7,nykoping,scandinavian_region,flatland,grasslands,continental,1.30,16.90,8.25,10.2,1.55,38.20
8,kolmarden,scandinavian_region,flatland,forest,continental,0.00,0.00,2.90,5.6,5.60,14.10
9,strangnas,scandinavian_region,flatland,woods,continental,0.00,0.00,3.00,6.2,4.20,13.40


In [7]:
print("Top 30 by total_rural_potential (with pop/dev):")
display(df_locations.nlargest(30, "total_rural_potential")[cols_full])
print("Top 30 by total_rural_potential_fixed (without pop/dev):")
display(df_locations.nlargest(30, "total_rural_potential_fixed")[cols_fixed])

Top 30 by total_rural_potential (with pop/dev):


,location,region,topography,vegetation,climate,fruit_orchard_potential,fishing_village_potential,sheep_farms_potential,farming_village_potential,forest_village_potential,total_rural_potential
8771,jiaxing,east_china_region,flatland,farmland,subtropical,43.4100,30.3870,37.7190,55.9980,0.0000,167.5140
8773,haiyan,east_china_region,flatland,farmland,subtropical,33.9750,21.6825,29.2275,45.3050,0.0000,130.1900
8619,changshu,east_china_region,flatland,farmland,subtropical,32.7000,22.8900,28.0800,43.8600,0.0000,127.5300
8599,liyang,east_china_region,flatland,farmland,subtropical,28.5750,17.9025,24.3675,44.2850,0.0000,115.1300
8772,chongde,east_china_region,flatland,farmland,subtropical,29.1000,18.2700,24.8400,39.7800,0.0000,111.9900
8631,jurong,east_china_region,flatland,farmland,subtropical,27.0300,16.8210,22.9770,42.5340,0.0000,109.3620
8607,ningguo,east_china_region,hills,woods,subtropical,26.6400,0.0000,22.6260,27.5280,30.1020,106.8960
8767,wuxing,east_china_region,flatland,farmland,subtropical,26.6250,16.5375,22.6125,36.9750,0.0000,102.7500
8630,jiangning,east_china_region,flatland,farmland,subtropical,25.9800,18.1860,22.0320,36.2440,0.0000,102.4420
8765,linan,east_china_region,hills,woods,subtropical,26.8050,0.0000,22.7745,27.6985,24.4615,101.7395


Top 30 by total_rural_potential_fixed (without pop/dev):


,location,region,topography,vegetation,climate,fruit_orchard_potential_fixed,fishing_village_potential_fixed,sheep_farms_potential_fixed,farming_village_potential_fixed,forest_village_potential_fixed,total_rural_potential_fixed
5678,split,balkan_region,hills,grasslands,mediterranean,9.60,16.20,11.20,10.38,1.40,48.78
7139,baniyas,crescent_region,hills,grasslands,mediterranean,9.60,16.20,11.20,10.38,1.40,48.78
3084,ajaccio,italy_region,hills,grasslands,mediterranean,9.60,5.40,16.00,15.57,1.40,47.97
2397,apt,france_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.40,46.05
2406,brignoles,france_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.40,46.05
5955,arta,balkan_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.40,46.05
17746,al_bayda,egypt_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.40,46.05
17970,thagia,maghreb_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.40,46.05
18346,paarl,southern_africa_region,hills,grasslands,mediterranean,18.60,0.00,11.20,14.85,1.40,46.05
3064,sabina,italy_region,hills,farmland,mediterranean,19.80,0.00,7.50,17.00,1.35,45.65


In [8]:
mult_cols = [f"local_{g}_output_multiplier" for g in goods]
display_cols = ["location", "region", "topography", "vegetation", "climate", "is_coastal", "has_river", "is_adjacent_to_lake"] + mult_cols
available = [c for c in display_cols if c in df_locations.columns]
df_locations[available].head(20)

,location,region,topography,vegetation,climate,is_coastal,has_river,is_adjacent_to_lake,local_fruit_output_multiplier,local_fish_output_multiplier,...,local_millet_output_multiplier,local_wheat_output_multiplier,local_maize_output_multiplier,local_rice_output_multiplier,local_legumes_output_multiplier,local_potato_output_multiplier,local_olives_output_multiplier,local_leather_output_multiplier,local_wild_game_output_multiplier,local_fur_output_multiplier
0,stockholm,scandinavian_region,flatland,grasslands,continental,yes,no,no,1.30,1.30,...,1.45,1.70,1.10,1.45,1.45,1.45,1.08,1.55,1.10,1.15
1,norrtalje,scandinavian_region,flatland,forest,continental,no,no,no,1.20,1.07,...,1.30,1.40,1.00,1.35,1.35,1.30,0.90,1.25,1.30,1.40
2,enkoping,scandinavian_region,flatland,grasslands,continental,no,no,no,1.25,1.05,...,1.45,1.70,1.10,1.40,1.45,1.45,1.00,1.55,1.10,1.15
3,uppsala,scandinavian_region,flatland,farmland,continental,no,no,no,1.35,1.05,...,1.40,1.75,1.15,1.50,1.55,1.50,1.05,1.50,0.75,0.95
4,kastelholm,scandinavian_region,flatland,grasslands,continental,yes,no,no,1.30,1.30,...,1.45,1.70,1.10,1.45,1.45,1.45,1.08,1.55,1.10,1.15
5,tierp,scandinavian_region,flatland,woods,continental,no,no,no,1.25,1.05,...,1.35,1.55,1.05,1.35,1.38,1.35,0.93,1.40,1.30,1.30
6,heby,scandinavian_region,flatland,grasslands,continental,no,no,no,1.25,1.05,...,1.45,1.70,1.10,1.40,1.45,1.45,1.00,1.55,1.10,1.15
7,nykoping,scandinavian_region,flatland,grasslands,continental,yes,no,no,1.30,1.30,...,1.45,1.70,1.10,1.45,1.45,1.45,1.08,1.55,1.10,1.15
8,kolmarden,scandinavian_region,flatland,forest,continental,no,no,no,1.20,1.07,...,1.30,1.40,1.00,1.35,1.35,1.30,0.90,1.25,1.30,1.40
9,strangnas,scandinavian_region,flatland,woods,continental,no,no,no,1.25,1.05,...,1.35,1.55,1.05,1.35,1.38,1.35,0.93,1.40,1.30,1.30


## Curiosities: high capacity, unusually low yield

Locations where a building has **high capacity** (top 25%) but **unusually low max yield** (bottom 25%)—the terrain gives lots of slots but poor output modifiers for that building's goods.

In [9]:
from IPython.display import display

CAPACITY_TOP_PCT = 50   # capacity percentile threshold (high = top 40%)
YIELD_BOTTOM_PCT = 50   # yield percentile threshold (low = bottom 40%)

rows = []
for b_name, building_id in BUILDING_NAME_TO_ID.items():
    if b_name not in df_cap_pivot.columns:
        continue
    cap_col = f"{building_id}_capacity"
    yield_col = f"{building_id}_potential"
    if yield_col not in df_locations.columns:
        continue

    # Capacity from pivot (Location -> value)
    cap_series = df_locations["location"].map(df_cap_pivot[b_name].to_dict()).fillna(0.0)
    yield_series = df_locations[yield_col]

    # Only buildable locations (capacity > 0)
    mask = cap_series > 0
    cap_valid = cap_series[mask]
    yield_valid = yield_series[mask]
    if cap_valid.empty:
        continue

    cap_pct = cap_valid.rank(pct=True) * 100
    yield_pct = yield_valid.rank(pct=True) * 100

    # Curiosity: high capacity rank, low yield rank
    curiosity_mask = (cap_pct >= CAPACITY_TOP_PCT) & (yield_pct <= YIELD_BOTTOM_PCT)
    for idx in cap_valid[curiosity_mask].index:
        loc = df_locations.loc[idx, "location"]
        rows.append({
            "location": loc,
            "building": b_name,
            "capacity": cap_series.loc[idx],
            "yield": yield_series.loc[idx],
            "capacity_pct": cap_pct.loc[idx],
            "yield_pct": yield_pct.loc[idx],
            "best_mult": yield_series.loc[idx] / cap_series.loc[idx] if cap_series.loc[idx] > 0 else 0,
            **{c: df_locations.loc[idx, c] for c in ["region", "topography", "vegetation", "climate"] if c in df_locations.columns},
        })

df_curiosities = pd.DataFrame(rows)
if df_curiosities.empty:
    # Fallback: show "near misses" — high capacity locations with lowest yield ratio
    rows_fallback = []
    for b_name, building_id in BUILDING_NAME_TO_ID.items():
        if b_name not in df_cap_pivot.columns:
            continue
        yield_col = f"{building_id}_potential"
        if yield_col not in df_locations.columns:
            continue
        cap_series = df_locations["location"].map(df_cap_pivot[b_name].to_dict()).fillna(0.0)
        yield_series = df_locations[yield_col]
        mask = cap_series > 0
        if mask.sum() < 2:
            continue
        cap_valid, yield_valid = cap_series[mask], yield_series[mask]
        # Yield/capacity ratio; rank ascending = lowest ratio first
        ratio = yield_valid / cap_valid
        low_ratio = ratio.nsmallest(5)
        for idx in low_ratio.index:
            rows_fallback.append({
                "location": df_locations.loc[idx, "location"],
                "building": b_name,
                "capacity": cap_series.loc[idx],
                "yield": yield_series.loc[idx],
                "best_mult": yield_series.loc[idx] / cap_series.loc[idx],
                **{c: df_locations.loc[idx, c] for c in ["region", "topography", "vegetation", "climate"] if c in df_locations.columns},
            })
    df_fallback = pd.DataFrame(rows_fallback)
    print("No curiosities with current thresholds. Showing locations with high capacity but lowest yield ratio per building:")
    display(df_fallback)
else:
    df_curiosities = df_curiosities.sort_values(["building", "capacity_pct"], ascending=[True, False]).reset_index(drop=True)
    display(df_curiosities)

,location,building,capacity,yield,capacity_pct,yield_pct,best_mult,region,topography,vegetation,climate
0,levanger,Farming Village,7.53,6.4005,75.311441,49.558464,0.85,scandinavian_region,hills,grasslands,arctic
1,tunanmarca,Farming Village,7.52,6.3920,75.248364,49.499330,0.85,andes_region,hills,grasslands,arctic
2,bitki,Farming Village,7.34,6.2390,73.971064,47.551841,0.85,manchuria_region,hills,grasslands,arctic
3,hatunqulla,Farming Village,7.23,6.1455,73.186549,46.771269,0.85,andes_region,hills,grasslands,arctic
4,hatun_xauxa,Farming Village,7.17,6.4530,72.784436,49.968462,0.90,andes_region,plateau,grasslands,arctic
...,...,...,...,...,...,...,...,...,...,...,...
2866,nukunu,Sheep Farm,5.01,5.5110,50.422691,47.401412,1.10,australia_region,hills,woods,cold_arid
2867,banggarla,Sheep Farm,5.01,5.5110,50.422691,47.401412,1.10,australia_region,hills,woods,cold_arid
2868,tidni,Sheep Farm,5.01,5.2605,50.422691,45.568951,1.05,australia_region,flatland,sparse,cold_arid
2869,bookabie,Sheep Farm,5.01,5.2605,50.422691,45.568951,1.05,australia_region,flatland,sparse,cold_arid
